# FEA LSTM Prediction Trajectories

This notebook generates `fea_prediction_trajectories.csv` for the final ordered FEA LSTM.

For each test sequence, the trained model is evaluated on prefixes containing the first 1, 2, ..., 30 observations. The resulting CSV can then be analyzed independently without rerunning model inference.

## 1. Setup

### 1.0 Add log filter

Add the log filter below or TensorFlow will print thousands of lines of uninformative log messages.

In [1]:
import re
import ipykernel.iostream

TF_LOG_FILTER_PATTERNS = [
    r'ptx\d+.*is not a recognized feature for this target',
    r'is not a recognized feature for this target \(ignoring feature\)',
    r'\(ignoring feature\)',
    r'successful NUMA node read from SysFS had negative value \(-1\)',
    r'gpu_timer\.cc:114\] Skipping the delay kernel, measurement accuracy will be reduced',
]

KERAS_PROGRESS_PATTERNS = [
    r'ms/step',
    r's/step',
    r'ETA:',
    r'\d+/\d+ \[',   # 12/64 [===>...]
]

_original_write = ipykernel.iostream.OutStream.write

def _filtered_write(self, msg, *args, **kwargs):
    text = str(msg)

    if any(re.search(p, text) for p in KERAS_PROGRESS_PATTERNS):
        _original_write(self, text, *args, **kwargs)
        return

    buf = getattr(self, '_tf_log_filter_buf', '')
    buf += text

    if '\n' not in buf:
        setattr(self, '_tf_log_filter_buf', buf)
        return

    lines = buf.splitlines(keepends=True)
    if not buf.endswith('\n'):
        incomplete = lines.pop()
    else:
        incomplete = ''

    for line in lines:
        if any(re.search(p, line) for p in TF_LOG_FILTER_PATTERNS):
            continue
        _original_write(self, line, *args, **kwargs)

    setattr(self, '_tf_log_filter_buf', incomplete)

ipykernel.iostream.OutStream.write = _filtered_write

print('Notebook log filter installed (targeted, keeps Keras steps).')

Notebook log filter installed (targeted, keeps Keras steps).


### 1.1 Imports and configuration

In [2]:
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from keras.layers import Input
from keras.models import Model, load_model


BATCH_SIZE = 32
SEED = 31
SEQUENCE_LENGTH = 30
N_FEATURES = 63

DATASET_PATH = Path('/workspace/datasets/emoji-hero-vr-db-dfea-as-csv')
TEST_PATH = DATASET_PATH / 'test_set.csv'
MODEL_PATH = Path('../models/fea_sequence_model.keras')
OUTPUT_PATH = Path('fea_prediction_trajectories.csv')

keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

for path in [TEST_PATH, MODEL_PATH]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

2026-09-19 12:53:51.625778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-19 12:53:51.634661: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-19 12:53:51.637278: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.17.0
Keras: 3.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/workspace/datasets/emoji-hero-vr-db-dfea-as-csv/test_set.csv: OK
../models/fea_sequence_model.keras: OK


In [3]:
ID_TO_EMOTION = {
    0: 'Anger',
    1: 'Disgust',
    2: 'Fear',
    3: 'Happiness',
    4: 'Neutral',
    5: 'Sadness',
    6: 'Surprise',
}

CLASS_NAMES = list(ID_TO_EMOTION.values())

## 2. Prepare the test sequences

Each sequence is sorted by its observation timestamp. In addition to the model input, sequence identifiers and timestamps are retained so prediction trajectories can later be aligned with the original FEA measurements.

In [4]:
test_df = pd.read_csv(TEST_PATH)

print('test_df.shape:', test_df.shape)
print('Columns:', test_df.columns.tolist())

feature_columns = test_df.columns[2:-1].tolist()
label_column = test_df.columns[-1]

assert len(feature_columns) == N_FEATURES
assert test_df['sequence_id'].nunique() == 378
assert (test_df.groupby('sequence_id').size() == SEQUENCE_LENGTH).all()
assert test_df[feature_columns].notna().all().all()

test_df.shape: (11340, 66)
Columns: ['sequence_id', 'timestamp', 'BrowLowererL', 'BrowLowererR', 'CheekPuffL', 'CheekPuffR', 'CheekRaiserL', 'CheekRaiserR', 'CheekSuckL', 'CheekSuckR', 'ChinRaiserB', 'ChinRaiserT', 'DimplerL', 'DimplerR', 'EyesClosedL', 'EyesClosedR', 'EyesLookDownL', 'EyesLookDownR', 'EyesLookLeftL', 'EyesLookLeftR', 'EyesLookRightL', 'EyesLookRightR', 'EyesLookUpL', 'EyesLookUpR', 'InnerBrowRaiserL', 'InnerBrowRaiserR', 'JawDrop', 'JawSidewaysLeft', 'JawSidewaysRight', 'JawThrust', 'LidTightenerL', 'LidTightenerR', 'LipCornerDepressorL', 'LipCornerDepressorR', 'LipCornerPullerL', 'LipCornerPullerR', 'LipFunnelerLB', 'LipFunnelerLT', 'LipFunnelerRB', 'LipFunnelerRT', 'LipPressorL', 'LipPressorR', 'LipPuckerL', 'LipPuckerR', 'LipStretcherL', 'LipStretcherR', 'LipSuckLB', 'LipSuckLT', 'LipSuckRB', 'LipSuckRT', 'LipTightenerL', 'LipTightenerR', 'LipsToward', 'LowerLipDepressorL', 'LowerLipDepressorR', 'MouthLeft', 'MouthRight', 'NoseWrinklerL', 'NoseWrinklerR', 'OuterBro

In [5]:
def prepare_data(df):
    sequences = []
    labels = []
    sequence_ids = []
    observation_timestamps = []

    for sequence_id, group in df.groupby('sequence_id'):
        group = group.sort_values('timestamp')

        sequences.append(group[feature_columns].to_numpy())
        labels.append(group.iloc[0][label_column])
        sequence_ids.append(str(sequence_id))
        observation_timestamps.append(group['timestamp'].to_numpy())

    return (
        np.asarray(sequences),
        np.asarray(labels),
        sequence_ids,
        np.asarray(observation_timestamps),
    )


X_test, y_test, sequence_ids, observation_timestamps = prepare_data(test_df)

assert X_test.shape == (378, SEQUENCE_LENGTH, N_FEATURES)
assert y_test.shape == (378,)
assert observation_timestamps.shape == (378, SEQUENCE_LENGTH)

print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_test: (378, 30, 63)
y_test: (378,)


In [6]:
def parse_sequence_id(sequence_id):
    parts = sequence_id.split('-')

    if len(parts) != 6:
        raise ValueError(f'Unexpected sequence_id format: {sequence_id}')

    sequence_timestamp, set_id, participant_id, level_id, emoji_id, emotion_id = parts

    return {
        'sequence_id': sequence_id,
        'sequence_timestamp': int(sequence_timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'emotion_id_from_sequence_id': int(emotion_id),
    }


sequence_metadata = pd.DataFrame([parse_sequence_id(sequence_id) for sequence_id in sequence_ids])

assert len(sequence_metadata) == 378
assert sequence_metadata['sequence_id'].is_unique
assert np.array_equal(sequence_metadata['emotion_id_from_sequence_id'].to_numpy(), y_test.astype(int))

sequence_metadata = sequence_metadata.drop(columns='emotion_id_from_sequence_id')

sequence_metadata.head()

,sequence_id,sequence_timestamp,set_id,participant_id,level_id,emoji_id
0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0
1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1
2,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2
3,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3
4,1700479005401-2-1-1-4-0,1700479005401,2,1,1,4


## 3. Load the trained model and build a prefix model

The prefix model reuses the trained layers and weights unchanged, but accepts variable sequence lengths. At 30 observations it must reproduce the original model exactly.

In [7]:
model = load_model(MODEL_PATH)

print('Input:', model.input_shape)
print('Output:', model.output_shape)

assert model.input_shape[1:] == (SEQUENCE_LENGTH, N_FEATURES)
assert model.output_shape[-1] == len(CLASS_NAMES)

model.summary()

Input: (None, 30, 63)
Output: (None, 7)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fea_input (InputLayer)          │ (None, 30, 63)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ norm_in (Normalization)         │ (None, 30, 63)         │           127 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ (None, 30, 63)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fea_dropout_1 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fea_dropout_2 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 347,286 (1.32 MB)

 Trainable params: 115,719 (452.03 KB)

 Non-trainable params: 127 (512.00 B)

 Optimizer params: 231,440 (904.07 KB)

In [8]:
prefix_input = Input(shape=(None, N_FEATURES), name='prefix_input')

x = prefix_input

# Reuse every trained layer except the original fixed-size InputLayer.
for layer in model.layers[1:]:
    x = layer(x)

prefix_model = Model(
    inputs=prefix_input,
    outputs=x,
    name='fea_prefix_model'
)

prefix_model.summary()

Model: "fea_prefix_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ prefix_input (InputLayer)       │ (None, None, 63)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ norm_in (Normalization)         │ (None, None, 63)       │           127 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ (None, None, 63)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fea_dropout_1 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fea_dropout_2 (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 115,846 (452.53 KB)

 Trainable params: 115,719 (452.03 KB)

 Non-trainable params: 127 (512.00 B)

In [9]:
full_sequence_probabilities = model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=0
)

prefix_probabilities_30 = prefix_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=0
)

assert np.allclose(
    full_sequence_probabilities,
    prefix_probabilities_30,
    atol=1e-6
)

print(
    'Maximum prediction difference:',
    np.max(np.abs(full_sequence_probabilities - prefix_probabilities_30))
)
print('Full-sequence predictions verified.')

Maximum prediction difference: 0.0
Full-sequence predictions verified.


## 4. Generate prediction trajectories

For timestep `t`, the model receives only observations `1 ... t` of each sequence. No padding or artificial observations are added.

In [10]:
n_sequences = len(X_test)
n_classes = len(CLASS_NAMES)

prediction_trajectories = np.empty(
    (n_sequences, SEQUENCE_LENGTH, n_classes),
    dtype=np.float32
)

for timestep in range(1, SEQUENCE_LENGTH + 1):
    X_prefix = X_test[:, :timestep, :]

    prediction_trajectories[:, timestep - 1, :] = prefix_model.predict(
        X_prefix,
        batch_size=BATCH_SIZE,
        verbose=0
    )

print('prediction_trajectories.shape:', prediction_trajectories.shape)

prediction_trajectories.shape: (378, 30, 7)


In [11]:
assert np.allclose(
    prediction_trajectories[:, -1, :],
    full_sequence_probabilities,
    atol=1e-6
)

assert np.allclose(
    prediction_trajectories.sum(axis=2),
    1.0,
    atol=1e-5
)

predicted_classes = np.argmax(prediction_trajectories, axis=2)

true_class_probabilities = np.take_along_axis(
    prediction_trajectories,
    y_test[:, None, None].astype(int),
    axis=2
).squeeze(axis=2)

final_predictions = predicted_classes[:, -1]
final_correct = final_predictions == y_test

print(f'Final accuracy: {final_correct.sum()}/378 = {final_correct.mean():.6f}')
assert final_correct.sum() == 296

Final accuracy: 296/378 = 0.783069


## 5. Export trajectory CSV

The output contains one row per sequence and prefix length (`378 × 30 = 11,340` rows). `sample_id` uniquely identifies each trajectory point as `<sequence_id>-<timestep>`, using timesteps `01` through `30`.

In [12]:
rows = []

for sequence_index in range(n_sequences):
    metadata = sequence_metadata.iloc[sequence_index].to_dict()
    true_class_id = int(y_test[sequence_index])

    for timestep_index in range(SEQUENCE_LENGTH):
        timestep = timestep_index + 1
        predicted_class_id = int(predicted_classes[sequence_index, timestep_index])

        row = {
            'sample_id': f"{metadata['sequence_id']}-{timestep:02d}",
            **metadata,
            'timestep': timestep,
            'observation_timestamp': observation_timestamps[sequence_index, timestep_index],
            'true_class_id': true_class_id,
            'true_class': ID_TO_EMOTION[true_class_id],
            'predicted_class_id': predicted_class_id,
            'predicted_class': ID_TO_EMOTION[predicted_class_id],
            'true_class_probability': true_class_probabilities[sequence_index, timestep_index],
            'predicted_class_probability': np.max(prediction_trajectories[sequence_index, timestep_index]),
            'correct_at_timestep': predicted_class_id == true_class_id,
            'correct_final_prediction': bool(final_correct[sequence_index]),
        }

        for class_id, class_name in ID_TO_EMOTION.items():
            row[f'probability_{class_name.lower()}'] = prediction_trajectories[
                sequence_index,
                timestep_index,
                class_id
            ]

        rows.append(row)

trajectory_df = pd.DataFrame(rows)

trajectory_df.head()

,sample_id,sequence_id,sequence_timestamp,set_id,participant_id,level_id,emoji_id,timestep,observation_timestamp,true_class_id,...,predicted_class_probability,correct_at_timestep,correct_final_prediction,probability_anger,probability_disgust,probability_fear,probability_happiness,probability_neutral,probability_sadness,probability_surprise
0,1700478995850-2-1-1-0-0-01,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,1700478994882,0,...,0.552614,True,True,0.552614,0.108155,0.023265,0.025473,0.027096,0.204593,0.058803
1,1700478995850-2-1-1-0-0-02,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,2,1700478994922,0,...,0.773931,True,True,0.773931,0.051726,0.003649,0.004114,0.004637,0.144593,0.017350
2,1700478995850-2-1-1-0-0-03,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,3,1700478994949,0,...,0.856087,True,True,0.856087,0.030706,0.001105,0.001200,0.001416,0.101567,0.007921
3,1700478995850-2-1-1-0-0-04,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,4,1700478994977,0,...,0.893805,True,True,0.893805,0.021771,0.000493,0.000502,0.000633,0.078230,0.004567
4,1700478995850-2-1-1-0-0-05,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,5,1700478995018,0,...,0.910399,True,True,0.910399,0.018251,0.000301,0.000288,0.000381,0.067024,0.003357


In [13]:
probability_columns = [
    f'probability_{class_name.lower()}'
    for class_name in CLASS_NAMES
]

assert len(trajectory_df) == 378 * SEQUENCE_LENGTH
assert trajectory_df['sample_id'].is_unique
assert trajectory_df['sequence_id'].nunique() == 378
assert trajectory_df.groupby('sequence_id').size().eq(SEQUENCE_LENGTH).all()
assert trajectory_df.groupby('sequence_id')['timestep'].nunique().eq(SEQUENCE_LENGTH).all()
assert trajectory_df.notna().all().all()

assert np.allclose(
    trajectory_df[probability_columns].sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.array_equal(
    trajectory_df['predicted_class_id'].to_numpy(),
    trajectory_df[probability_columns].to_numpy().argmax(axis=1)
)

final_rows = trajectory_df[trajectory_df['timestep'] == SEQUENCE_LENGTH]
assert len(final_rows) == 378
assert final_rows['correct_at_timestep'].sum() == 296

print('Trajectory table passed all integrity checks.')

Trajectory table passed all integrity checks.


In [14]:
trajectory_df.to_csv(OUTPUT_PATH, index=False)

print(f'Exported {len(trajectory_df):,} rows to {OUTPUT_PATH.resolve()}')
print(f'Columns: {len(trajectory_df.columns)}')

Exported 11,340 rows to /workspace/repos/emohevrdb-dfer/5_dynamic_facial_expression_recognition/5_2_fea_sequence_based_fer/5_2_4_sequence_order_ablation/trajectories/fea_prediction_trajectories.csv
Columns: 24
